# Where each global income decile lives

For each indicator, the bars show the share of each global income decile that lives in countries where **fewer than half the population** has the service (or, for the last panel, in countries on the World Bank's fragile and conflict-affected list).

This is a country-level measure: it shows where people live, not which households lack a service. Adult literacy is left out because its country observations come from different years.

In [52]:
import sys
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd
from IPython.display import display

NOTEBOOK_DIR = Path.cwd() if (Path.cwd() / 'indicator_analysis_helpers.py').exists() else Path.cwd() / 'dev_indicators'
sys.path.insert(0, str(NOTEBOOK_DIR))
from indicator_analysis_helpers import D1_FILE, fetch_indicator
from parse_fcs_list import parse_fcs

THRESHOLD = 50
D1_PATH = D1_FILE if D1_FILE.exists() else NOTEBOOK_DIR.parent / D1_FILE.name
FCS_PDF = NOTEBOOK_DIR.parent / 'data' / 'C2-FCSList-FY06toFY26.pdf'

distribution = pd.read_csv(D1_PATH)
for column in ('year', 'welf', 'pop'):
    distribution[column] = pd.to_numeric(distribution[column], errors='coerce')
distribution = distribution.dropna(subset=['year', 'welf', 'pop'])


def global_deciles(year):
    bins = distribution[distribution['year'].eq(year)].sort_values('welf', kind='stable').copy()
    bins['decile'] = np.minimum(
        np.ceil(bins['pop'].cumsum() / (bins['pop'].sum() / 10)), 10
    ).astype(int)
    return bins


def decile_shares(bins, flagged):
    # flagged maps country code -> 1.0 (in group) / 0.0; codes without data map to NaN.
    bins = bins.assign(flag=bins['code'].map(flagged))
    rows = []
    for decile, group in bins.groupby('decile'):
        observed = group.dropna(subset=['flag'])
        rows.append({
            'decile': decile,
            'share': 100 * observed.loc[observed['flag'].eq(1), 'pop'].sum() / observed['pop'].sum(),
            'coverage': 100 * observed['pop'].sum() / group['pop'].sum(),
        })
    return pd.DataFrame(rows)


SERVICES = [
    ('Most people lack electricity', 'WB_WDI', 'WB_WDI_EG_ELC_ACCS_ZS', 2023),
    ("Most people don't use the internet", 'ITU_DH', 'IT_NET_USER', 2024),
    ('Most people lack basic sanitation', 'WB_WDI', 'WB_WDI_SH_STA_BASS_ZS', 2024),
]
frames = []
for label, dataset, indicator, year in SERVICES:
    values = fetch_indicator(dataset, indicator)
    values = values[values['year'].eq(year)].drop_duplicates('code').set_index('code')['indicator_value']
    shares = decile_shares(global_deciles(year), values.lt(THRESHOLD).astype(float))
    frames.append(shares.assign(panel=f'{label} ({year})'))

fcs_set = set(parse_fcs(FCS_PDF).query('fiscal_year == 2026')['code'])
fcs_bins = global_deciles(2025)
fcs_codes = fcs_bins['code'].unique()
fcs_flag = pd.Series([1.0 if code in fcs_set else 0.0 for code in fcs_codes], index=fcs_codes)
frames.append(decile_shares(fcs_bins, fcs_flag).assign(panel='Fragile or conflict-affected state (2025)'))

chart_data = pd.concat(frames, ignore_index=True)
chart_data['decile_label'] = 'D' + chart_data['decile'].astype(str)
PANEL_ORDER = [frame['panel'].iloc[0] for frame in frames]

display(
    chart_data.pivot(index='decile_label', columns='panel', values='share')
    .reindex([f'D{i}' for i in range(1, 11)])[PANEL_ORDER]
    .round(2)
)
print('Minimum population coverage of the indicator data across deciles, by panel (%):')
display(chart_data.groupby('panel')['coverage'].min().reindex(PANEL_ORDER).round(1))

panel,Most people lack electricity (2023),Most people don't use the internet (2024),Most people lack basic sanitation (2024),Fragile or conflict-affected state (2025)
decile_label,,,,
D1,30.57,74.00,66.33,55.76
D2,7.67,33.42,29.78,28.13
D3,3.42,16.40,14.73,14.45
D4,2.24,11.82,10.51,10.26
D5,1.54,8.80,7.60,8.39
D6,1.09,6.20,5.21,7.37
D7,0.62,3.54,2.94,5.82
D8,0.31,1.47,1.20,3.20
D9,0.12,0.43,0.37,1.22


Minimum population coverage of the indicator data across deciles, by panel (%):


panel
Most people lack electricity (2023)           98.7
Most people don't use the internet (2024)     90.6
Most people lack basic sanitation (2024)      97.1
Fragile or conflict-affected state (2025)    100.0
Name: coverage, dtype: float64

## Chart

All panels share the same vertical scale. D1 is highlighted and labeled.

In [53]:
DECILE_ORDER = [f'D{i}' for i in range(1, 11)]

base = alt.Chart().encode(
    x=alt.X('decile_label:N', sort=DECILE_ORDER, title='Global income decile (D1 = poorest)', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('share:Q', title='Share of decile living there (%)', scale=alt.Scale(domain=[0, 80])),
)
bars = base.mark_bar().encode(
    color=alt.condition(alt.datum.decile == 1, alt.value('#D55E00'), alt.value('#BDBDBD')),
    tooltip=[
        alt.Tooltip('panel:N', title='Countries where'),
        alt.Tooltip('decile_label:N', title='Decile'),
        alt.Tooltip('share:Q', title='Share of decile (%)', format='.1f'),
        alt.Tooltip('coverage:Q', title='Decile population with data (%)', format='.1f'),
    ],
)
d1_labels = (
    base.transform_filter(alt.datum.decile == 1)
    .transform_calculate(label="format(datum.share, '.0f') + '%'")
    .mark_text(dy=-7, fontSize=12, fontWeight='bold', color='#D55E00')
    .encode(text='label:N')
)

decile_bars = (
    alt.layer(bars, d1_labels, data=chart_data)
    .properties(width=280, height=190)
    .facet(
        facet=alt.Facet('panel:N', sort=PANEL_ORDER, title=None, header=alt.Header(labelFontSize=13, labelFontWeight='bold')),
        columns=2,
    )
    .properties(
        title=alt.TitleParams(
            'The poorest tenth is concentrated where basic services have not arrived',
            subtitle=['Share of each global income decile living in countries where fewer than half the population has each service, or in fragile/conflict-affected states'],
            anchor='start',
            fontSize=16,
        )
    )
    .configure_view(stroke=None)
    .configure_axis(grid=False)
)
decile_bars

alt.FacetChart(...)

## Suggested caption and note

**Caption:** In every panel, the poorest decile stands apart: 31% live in countries where most people lack electricity, 74% in countries where most people do not use the internet and 66% in countries where most people lack basic sanitation, compared with almost none of the richest decile. More than half live in fragile or conflict-affected states.

**Note:** Country-level classification; it shows where people live, not which households lack each service. Sources: global income distribution (1,000 bins per country); WDI electricity access; ITU internet use; WHO/UNICEF JMP basic sanitation via WDI; World Bank FY26 list of fragile and conflict-affected situations. Years differ by indicator.

# Has this changed since 2000?

Same measure, computed for around 2000 and for the latest year. Global deciles are rebuilt for each year. The fragility list starts in FY2006, so its baseline is 2005. The list's methodology changed in FY2009 and FY2020, so its comparison is indicative only.

Two things can move at once:
- **The absolute share:** how much of D1 lives in low-coverage countries.
- **The relative concentration:** D1's share divided by the share of the whole world population living there. The maximum is 10x, which would mean everyone in those countries is in D1.

In [54]:
COMPARISONS = [
    ('Electricity', 'WB_WDI', 'WB_WDI_EG_ELC_ACCS_ZS', 2000, 2023),
    ('Internet', 'ITU_DH', 'IT_NET_USER', 2000, 2024),
    ('Sanitation', 'WB_WDI', 'WB_WDI_SH_STA_BASS_ZS', 2000, 2024),
]


def world_share(bins, flagged):
    observed = bins.assign(flag=bins['code'].map(flagged)).dropna(subset=['flag'])
    return 100 * observed.loc[observed['flag'].eq(1), 'pop'].sum() / observed['pop'].sum()


def period_rows(indicator, year, period, bins, flagged):
    shares = decile_shares(bins, flagged)
    return shares.assign(indicator=indicator, year=year, period=period, world=world_share(bins, flagged))


change_frames = []
for indicator, dataset, indicator_id, start, end in COMPARISONS:
    values = fetch_indicator(dataset, indicator_id).drop_duplicates(['code', 'year'])
    for year, period in ((start, 'Around 2000'), (end, 'Latest')):
        flagged = values[values['year'].eq(year)].set_index('code')['indicator_value'].lt(THRESHOLD).astype(float)
        change_frames.append(period_rows(indicator, year, period, global_deciles(year), flagged))

fcs_list = parse_fcs(FCS_PDF)
for fiscal_year, period in ((2006, 'Around 2000'), (2026, 'Latest')):
    fcs_codes_year = set(fcs_list.loc[fcs_list['fiscal_year'].eq(fiscal_year), 'code'])
    bins = global_deciles(fiscal_year - 1)
    codes = bins['code'].unique()
    flagged = pd.Series([1.0 if code in fcs_codes_year else 0.0 for code in codes], index=codes)
    change_frames.append(period_rows('Fragile or conflict-affected', fiscal_year - 1, period, bins, flagged))

change_data = pd.concat(change_frames, ignore_index=True)
change_data['decile_label'] = 'D' + change_data['decile'].astype(str)
change_data['period_label'] = change_data['period'] + ' (' + change_data['year'].astype(str) + ')'

d1_change = change_data[change_data['decile'].eq(1)].assign(ratio=lambda df: df['share'] / df['world'])
change_table = d1_change.pivot(index='indicator', columns='period', values=['year', 'share', 'world', 'ratio'])
change_table.columns = [f'{metric} ({period})' for metric, period in change_table.columns]
INDICATOR_ORDER = [row[0] for row in COMPARISONS] + ['Fragile or conflict-affected']
display(change_table.reindex(INDICATOR_ORDER).round(2))
print('Minimum population coverage across deciles (%):')
display(change_data.groupby(['indicator', 'period'])['coverage'].min().unstack().reindex(INDICATOR_ORDER).round(1))

,year (Around 2000),year (Latest),share (Around 2000),share (Latest),world (Around 2000),world (Latest),ratio (Around 2000),ratio (Latest)
indicator,,,,,,,,
Electricity,2000.0,2023.0,38.71,30.57,14.71,4.77,2.63,6.40
Internet,2000.0,2024.0,99.99,74.00,99.40,15.02,1.01,4.93
Sanitation,2000.0,2024.0,47.25,66.33,40.09,13.85,1.18,4.79
Fragile or conflict-affected,2005.0,2025.0,26.26,55.76,7.63,13.49,3.44,4.13


Minimum population coverage across deciles (%):


period,Around 2000,Latest
indicator,,
Electricity,97.9,98.7
Internet,93.2,90.6
Sanitation,89.6,97.1
Fragile or conflict-affected,100.0,100.0


In [55]:
ratio_lookup = d1_change.set_index(['indicator', 'period'])['ratio']
change_data['panel'] = change_data['indicator'].map(
    lambda name: f"{name}: D1 {ratio_lookup[(name, 'Around 2000')]:.1f}x → {ratio_lookup[(name, 'Latest')]:.1f}x the world"
)
CHANGE_PANEL_ORDER = [change_data.loc[change_data['indicator'].eq(name), 'panel'].iloc[0] for name in INDICATOR_ORDER]

change_bars = (
    alt.Chart(change_data)
    .mark_bar()
    .encode(
        x=alt.X('decile_label:N', sort=DECILE_ORDER, title='Global income decile (D1 = poorest)', axis=alt.Axis(labelAngle=0)),
        xOffset=alt.XOffset('period:N', sort=['Around 2000', 'Latest']),
        y=alt.Y('share:Q', title='Share of decile living there (%)', scale=alt.Scale(domain=[0, 100])),
        color=alt.Color(
            'period:N',
            sort=['Around 2000', 'Latest'],
            scale=alt.Scale(domain=['Around 2000', 'Latest'], range=['#F4C7A1', '#D55E00']),
            legend=alt.Legend(title=None, orient='top'),
        ),
        tooltip=[
            alt.Tooltip('indicator:N', title='Indicator'),
            alt.Tooltip('decile_label:N', title='Decile'),
            alt.Tooltip('period_label:N', title='Year'),
            alt.Tooltip('share:Q', title='Share of decile (%)', format='.1f'),
            alt.Tooltip('world:Q', title='Share of world (%)', format='.1f'),
            alt.Tooltip('coverage:Q', title='Decile population with data (%)', format='.1f'),
        ],
    )
    .properties(width=300, height=190)
    .facet(
        facet=alt.Facet('panel:N', sort=CHANGE_PANEL_ORDER, title=None, header=alt.Header(labelFontSize=12, labelFontWeight='bold')),
        columns=2,
    )
    .properties(
        title=alt.TitleParams(
            'As the rest of the world gained access, the poorest tenth was left behind',
            subtitle=[
                'Share of each global income decile living in countries where fewer than half the population has each service,',
                'or in fragile/conflict-affected states. Baseline 2000 (fragility: 2005); latest 2023-2025.',
            ],
            anchor='start',
            fontSize=16,
        )
    )
    .configure_view(stroke=None)
    .configure_axis(grid=False)
)
change_bars

alt.FacetChart(...)

## Reading the change

- **Electricity and internet:** the share of D1 in low-coverage countries **fell**, but slowly. For everyone else it fell much faster, almost to zero. What remains is increasingly a D1 problem, so D1's concentration relative to the world rose.
- **Internet in 2000:** almost every country was below 50%, so every decile sits near 100%. The change is the rest of the distribution moving out while D1 stays.
- **Sanitation:** D1 is the only decile whose share **rose**, from 47% to 66%. In 2000 D1 was less exposed than D2-D5; today it is more than twice as exposed as D2.
- **Fragility:** D1's absolute share also rose, from about a quarter to more than half. Part of this may reflect changes in the list's methodology.

**Suggested framing:** "Progress has largely bypassed the poorest tenth. On sanitation and fragility, it has gone backwards."

# Slope chart: who is left in low-coverage countries?

This flips the question. Among all people living in low-coverage countries, what share belongs to the world's poorest decile? If D1 were spread like everyone else, that share would be 10%. It is equal to the D1/world ratio divided by 10, up to small rounding because the global deciles are built from discrete bins.

In [56]:
SLOPE_COLORS = {
    'Electricity': '#D55E00',
    'Internet': '#0072B2',
    'Sanitation': '#009E73',
    'Fragile or conflict-affected': '#CC79A7',
}
PERIOD_LABELS = {'Around 2000': 'Around 2000', 'Latest': 'Today (2023-2025)'}

slope_data = d1_change[['indicator', 'period', 'year', 'share', 'world']].copy()
# Share of the low-coverage population in D1 = D1 share x D1 population / (world share x total population).
d1_population_fraction = {
    year: bins.loc[bins['decile'].eq(1), 'pop'].sum() / bins['pop'].sum()
    for year, bins in ((year, global_deciles(year)) for year in slope_data['year'].unique())
}
slope_data['d1_part'] = slope_data['share'] * slope_data['year'].map(d1_population_fraction) / slope_data['world'] * 100
display(slope_data.pivot(index='indicator', columns='period', values='d1_part').reindex(INDICATOR_ORDER).round(1))
slope_data['period_axis'] = slope_data['period'].map(PERIOD_LABELS)


def spread_labels(values, min_gap=4):
    # Nudge label positions apart so close values do not overlap.
    order = values.sort_values().index
    positions = values.copy()
    for previous, current in zip(order[:-1], order[1:]):
        positions[current] = max(positions[current], positions[previous] + min_gap)
    return positions


slope_data['label_y'] = slope_data.groupby('period', group_keys=False)['d1_part'].apply(spread_labels)
slope_data['start_label'] = slope_data['d1_part'].round().astype(int).astype(str) + '%'
slope_data['end_label'] = slope_data['start_label'] + '  ' + slope_data['indicator']
slope_data.loc[slope_data['indicator'].eq('Fragile or conflict-affected'), 'end_label'] += ' (2005 → 2025)'

SLOPE_WIDTH = 420
x_axis = alt.X(
    'period_axis:N', sort=list(PERIOD_LABELS.values()), title=None,
    axis=alt.Axis(labelAngle=0, labelFontSize=13, labelFontWeight='bold', domain=False, ticks=False, orient='top'),
    scale=alt.Scale(padding=0.35),
)
y_scale = alt.Scale(domain=[0, 70])
color = alt.Color('indicator:N', scale=alt.Scale(domain=list(SLOPE_COLORS), range=list(SLOPE_COLORS.values())), legend=None)

slope_base = alt.Chart(slope_data).encode(x=x_axis, color=color)
lines = slope_base.mark_line(strokeWidth=3).encode(
    y=alt.Y(
        'd1_part:Q',
        scale=y_scale,
        axis=alt.Axis(title='Share of low-coverage population in D1 (%)', values=list(range(0, 71, 10)), grid=False),
    ),
    strokeDash=alt.condition(
        alt.datum.indicator == 'Fragile or conflict-affected', alt.value([5, 4]), alt.value([1, 0])
    ),
    detail='indicator:N',
)
points = slope_base.mark_point(filled=True, size=70, opacity=1).encode(
    y=alt.Y('d1_part:Q', scale=y_scale, axis=None),
    tooltip=[
        alt.Tooltip('indicator:N', title='Indicator'),
        alt.Tooltip('year:Q', title='Year', format='d'),
        alt.Tooltip('d1_part:Q', title='Share who are in D1 (%)', format='.1f'),
    ],
)
start_text = slope_base.transform_filter(alt.datum.period == 'Around 2000').mark_text(
    align='right', dx=-10, fontSize=12, fontWeight='bold'
).encode(y=alt.Y('label_y:Q', scale=y_scale, axis=None), text='start_label:N')
end_text = slope_base.transform_filter(alt.datum.period == 'Latest').mark_text(
    align='left', dx=10, fontSize=12, fontWeight='bold'
).encode(y=alt.Y('label_y:Q', scale=y_scale, axis=None), text='end_label:N')

reference = alt.Chart(pd.DataFrame({'y': [10]}))
reference_rule = reference.mark_rule(strokeDash=[2, 3], color='#7F7F7F').encode(y=alt.Y('y:Q', scale=y_scale, axis=None))
reference_text = reference.mark_text(
    text='10% = proportional representation', align='center', dy=-7, fontSize=11, color='#7F7F7F',
).encode(x=alt.value(SLOPE_WIDTH / 2), y=alt.Y('y:Q', scale=y_scale, axis=None))

slope_chart = (
    alt.layer(reference_rule, reference_text, lines, points, start_text, end_text)
    .properties(
        width=SLOPE_WIDTH,
        height=360,
        title=alt.TitleParams(
            'Low-coverage countries are increasingly home to the poorest tenth',
            subtitle=[
                'Share of people living in countries where fewer than half the population has each service',
                "(or in fragile/conflict-affected states) who belong to the world's poorest income decile",
            ],
            anchor='start',
            fontSize=16,
            offset=12,
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=False)
)
slope_chart

period,Around 2000,Latest
indicator,,
Electricity,26.3,64.0
Internet,10.1,49.2
Sanitation,11.8,47.9
Fragile or conflict-affected,34.4,41.3


alt.LayerChart(...)

**Suggested caption:** In 2000, about one in four people living in countries where most lacked electricity belonged to the world's poorest tenth; today it is almost two in three. For internet use and basic sanitation, the share rose from about one in ten to about half.

**Note:** Country-level classification; it does not identify which households lack each service. The fragility series (dashed) starts in 2005, and the World Bank list changed methodology in FY2009 and FY2020.

In [57]:
services_slope = slope_data[slope_data['indicator'].ne('Fragile or conflict-affected')].copy()
services_slope['period_axis'] = services_slope['period'].map({'Around 2000': '2000', 'Latest': 'Today'})
services_slope['label_y'] = services_slope.groupby('period', group_keys=False)['d1_part'].apply(spread_labels)
services_slope['end_label'] = services_slope['start_label'] + '  ' + services_slope['indicator']

services_x = alt.X(
    'period_axis:N', sort=['2000', 'Today'], title=None,
    axis=alt.Axis(labelAngle=0, labelFontSize=13, labelFontWeight='bold', domain=False, ticks=False, orient='top'),
    scale=alt.Scale(padding=0.35),
)
services_base = alt.Chart(services_slope).encode(x=services_x, color=color)
services_lines = services_base.mark_line(strokeWidth=3).encode(
    y=alt.Y(
        'd1_part:Q',
        scale=y_scale,
        axis=alt.Axis(title='Share of low-coverage population in D1 (%)', values=list(range(0, 71, 10)), grid=False),
    ),
    detail='indicator:N',
)
services_points = services_base.mark_point(filled=True, size=70, opacity=1).encode(
    y=alt.Y('d1_part:Q', scale=y_scale, axis=None),
    tooltip=[
        alt.Tooltip('indicator:N', title='Indicator'),
        alt.Tooltip('year:Q', title='Year', format='d'),
        alt.Tooltip('d1_part:Q', title='Share who are in D1 (%)', format='.1f'),
    ],
)
services_start = services_base.transform_filter(alt.datum.period == 'Around 2000').mark_text(
    align='right', dx=-10, fontSize=12, fontWeight='bold'
).encode(y=alt.Y('label_y:Q', scale=y_scale, axis=None), text='start_label:N')
services_end = services_base.transform_filter(alt.datum.period == 'Latest').mark_text(
    align='left', dx=10, fontSize=12, fontWeight='bold'
).encode(y=alt.Y('label_y:Q', scale=y_scale, axis=None), text='end_label:N')

services_slope_chart = (
    alt.layer(reference_rule, reference_text, services_lines, services_points, services_start, services_end)
    .properties(
        width=SLOPE_WIDTH,
        height=360,
        title=alt.TitleParams(
            'Low-coverage countries are increasingly home to the poorest tenth',
            subtitle=[
                'Share of people living in countries where fewer than half the population has each service',
                "who belong to the world's poorest income decile",
            ],
            anchor='start',
            fontSize=16,
            offset=12,
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(grid=False)
)
services_slope_chart

alt.LayerChart(...)

In [58]:
decile_population_fraction = {}
for year in change_data['year'].unique():
    bins = global_deciles(year)
    fractions = bins.groupby('decile')['pop'].sum() / bins['pop'].sum()
    for decile in (2, 5):
        decile_population_fraction[(int(year), decile)] = fractions.loc[decile]

services_by_decile = change_data[
    change_data['indicator'].isin(['Electricity', 'Internet', 'Sanitation'])
    & change_data['decile'].isin([2, 5])
].copy()
services_by_decile['population_fraction'] = [
    decile_population_fraction[(int(year), int(decile))]
    for year, decile in zip(services_by_decile['year'], services_by_decile['decile'])
]
services_by_decile['decile_part'] = (
    services_by_decile['share']
    * services_by_decile['population_fraction']
    / services_by_decile['world']
    * 100
)


def make_decile_services_chart(decile):
    data = services_by_decile[services_by_decile['decile'].eq(decile)].copy()
    data['period_axis'] = data['period'].map({'Around 2000': '2000', 'Latest': 'Today'})
    data['label_y'] = data.groupby('period', group_keys=False)['decile_part'].apply(spread_labels)
    data['value_label'] = data['decile_part'].round().astype(int).astype(str) + '%'
    data['end_label'] = data['value_label'] + '  ' + data['indicator']

    base = alt.Chart(data).encode(x=services_x, color=color)
    lines = base.mark_line(strokeWidth=3).encode(
        y=alt.Y(
            'decile_part:Q',
            scale=y_scale,
            axis=alt.Axis(
                title=f'Share of low-coverage population in D{decile} (%)',
                values=list(range(0, 71, 10)),
                grid=False,
            ),
        ),
        detail='indicator:N',
    )
    points = base.mark_point(filled=True, size=70, opacity=1).encode(
        y=alt.Y('decile_part:Q', scale=y_scale, axis=None),
        tooltip=[
            alt.Tooltip('indicator:N', title='Indicator'),
            alt.Tooltip('year:Q', title='Year', format='d'),
            alt.Tooltip('decile_part:Q', title=f'Share who are in D{decile} (%)', format='.1f'),
        ],
    )
    start = base.transform_filter(alt.datum.period == 'Around 2000').mark_text(
        align='right', dx=-10, fontSize=12, fontWeight='bold'
    ).encode(y=alt.Y('label_y:Q', scale=y_scale, axis=None), text='value_label:N')
    end = base.transform_filter(alt.datum.period == 'Latest').mark_text(
        align='left', dx=10, fontSize=12, fontWeight='bold'
    ).encode(y=alt.Y('label_y:Q', scale=y_scale, axis=None), text='end_label:N')

    return (
        alt.layer(reference_rule, reference_text, lines, points, start, end)
        .properties(
            width=SLOPE_WIDTH,
            height=360,
            title=alt.TitleParams(
                f'Who belongs to global income decile {decile}?',
                subtitle=[
                    'Share of people living in countries where fewer than half the population has each service',
                    f'who belong to D{decile}; baseline 2000 and latest 2023-2024',
                ],
                anchor='start',
                fontSize=16,
                offset=12,
            ),
        )
        .configure_view(stroke=None)
        .configure_axis(grid=False)
    )


decile_2_slope_chart = make_decile_services_chart(2)
decile_5_slope_chart = make_decile_services_chart(5)
display(decile_2_slope_chart)
display(decile_5_slope_chart)

alt.LayerChart(...)

alt.LayerChart(...)

In [59]:
conditional_data = change_data[
    change_data['indicator'].isin(['Electricity', 'Internet', 'Sanitation'])
    & change_data['decile'].isin([1, 2, 5])
].copy()
conditional_data['series'] = conditional_data['decile_label']
conditional_data['probability'] = conditional_data['share']

world_reference = conditional_data[
    ['indicator', 'year', 'period', 'world']
].drop_duplicates().rename(columns={'world': 'probability'})
world_reference['series'] = 'World population'

conditional_chart_data = pd.concat(
    [
        conditional_data[['indicator', 'year', 'period', 'series', 'probability']],
        world_reference[['indicator', 'year', 'period', 'series', 'probability']],
    ],
    ignore_index=True,
)
conditional_chart_data['period_axis'] = conditional_chart_data['period'].map(
    {'Around 2000': '2000', 'Latest': 'Latest'}
)

CONDITIONAL_SERIES = ['D1', 'D2', 'D5', 'World population']
CONDITIONAL_COLORS = ['#D55E00', '#0072B2', '#009E73', '#7F7F7F']
conditional_base = alt.Chart(conditional_chart_data).encode(
    x=alt.X(
        'period_axis:N',
        sort=['2000', 'Latest'],
        title=None,
        axis=alt.Axis(labelAngle=0, labelFontWeight='bold', domain=False, ticks=False),
    ),
    y=alt.Y(
        'probability:Q',
        title='Share living in low-access countries (%)',
        scale=alt.Scale(domain=[0, 100]),
        axis=alt.Axis(values=list(range(0, 101, 20)), grid=True),
    ),
    color=alt.Color(
        'series:N',
        title=None,
        sort=CONDITIONAL_SERIES,
        scale=alt.Scale(domain=CONDITIONAL_SERIES, range=CONDITIONAL_COLORS),
        legend=alt.Legend(orient='top'),
    ),
    detail='series:N',
)
conditional_lines = conditional_base.mark_line().encode(
    strokeWidth=alt.condition(alt.datum.series == 'D1', alt.value(3), alt.value(2)),
    strokeDash=alt.condition(alt.datum.series == 'World population', alt.value([4, 3]), alt.value([1, 0])),
)
conditional_points = conditional_base.mark_point(filled=True, size=65).encode(
    tooltip=[
        alt.Tooltip('indicator:N', title='Service'),
        alt.Tooltip('series:N', title='Group'),
        alt.Tooltip('year:Q', title='Year', format='d'),
        alt.Tooltip('probability:Q', title='Share (%)', format='.1f'),
    ],
)

conditional_probability_chart = (
    alt.layer(conditional_lines, conditional_points)
    .facet(
        facet=alt.Facet(
            'indicator:N',
            sort=['Electricity', 'Internet', 'Sanitation'],
            title=None,
            header=alt.Header(labelFontSize=13, labelFontWeight='bold'),
        ),
        columns=3,
    )
    .properties(
        title=alt.TitleParams(
            'How likely is each decile to live in a low-access country?',
            subtitle=[
                'Population-weighted share by decile; latest year is 2023 for electricity, 2024 for internet and sanitation.',
                'Dashed line is the world population share; decile share divided by this line gives the relative concentration (x).',
            ],
            anchor='start',
            fontSize=16,
        ),
    )
    .configure_view(stroke=None)
    .configure_axis(gridColor='#E6E6E6', gridOpacity=0.5)
)
conditional_probability_chart

alt.FacetChart(...)

In [60]:
def group_share(bins, flagged, deciles):
    observed = bins[bins['decile'].isin(deciles)].assign(flag=lambda df: df['code'].map(flagged)).dropna(subset=['flag'])
    return 100 * observed.loc[observed['flag'].eq(1), 'pop'].sum() / observed['pop'].sum()


latest_flags = []
for indicator, dataset, indicator_id, _, end in COMPARISONS:
    values = fetch_indicator(dataset, indicator_id).drop_duplicates(['code', 'year'])
    flagged = values[values['year'].eq(end)].set_index('code')['indicator_value'].lt(THRESHOLD).astype(float)
    latest_flags.append((indicator, end, flagged))

fcs_bins_latest = global_deciles(2025)
fcs_codes_latest = set(fcs_list.loc[fcs_list['fiscal_year'].eq(2026), 'code'])
fcs_flagged = pd.Series(
    [1.0 if code in fcs_codes_latest else 0.0 for code in fcs_bins_latest['code'].unique()],
    index=fcs_bins_latest['code'].unique(),
)
latest_flags.append(('Fragile or conflict-affected', 2025, fcs_flagged))

rows = []
for indicator, year, flagged in latest_flags:
    bins = global_deciles(year)
    d1 = group_share(bins, flagged, [1])
    d2 = group_share(bins, flagged, [2])
    rest = group_share(bins, flagged, range(2, 11))
    world = group_share(bins, flagged, range(1, 11))
    rows.append({
        'indicator': indicator, 'year': year, 'D1': d1, 'D2': d2, 'D2-D10': rest, 'World': world,
        'D1/D2': d1 / d2, 'D1/(D2-D10)': d1 / rest, 'D1/World': d1 / world,
    })

blog_numbers = pd.DataFrame(rows).set_index('indicator')
blog_numbers.round(2)

,year,D1,D2,D2-D10,World,D1/D2,D1/(D2-D10),D1/World
indicator,,,,,,,,
Electricity,2023,30.57,7.67,1.90,4.77,3.98,16.11,6.40
Internet,2024,74.00,33.42,8.96,15.02,2.21,8.26,4.93
Sanitation,2024,66.33,29.78,8.07,13.85,2.23,8.22,4.79
Fragile or conflict-affected,2025,55.76,28.13,8.80,13.49,1.98,6.34,4.13


In [61]:
start_flags = []
for indicator, dataset, indicator_id, start, _ in COMPARISONS:
    values = fetch_indicator(dataset, indicator_id).drop_duplicates(['code', 'year'])
    flagged = values[values['year'].eq(start)].set_index('code')['indicator_value'].lt(THRESHOLD).astype(float)
    start_flags.append((indicator, start, flagged))

fcs_bins_start = global_deciles(2005)
fcs_codes_start = set(fcs_list.loc[fcs_list['fiscal_year'].eq(2006), 'code'])
start_flags.append((
    'Fragile or conflict-affected', 2005,
    pd.Series({code: float(code in fcs_codes_start) for code in fcs_bins_start['code'].unique()}),
))


def d1_people(bins, flagged):
    return bins.loc[bins['decile'].eq(1) & bins['code'].map(flagged).eq(1), 'pop'].sum()


rows = []
for (indicator, y0, f0), (_, y1, f1) in zip(start_flags, latest_flags):
    b0, b1 = global_deciles(y0), global_deciles(y1)
    rows.append({
        'indicator': indicator, 'start': y0, 'end': y1,
        'D1 start': group_share(b0, f0, [1]), 'D1 end': group_share(b1, f1, [1]),
        'D2 start': group_share(b0, f0, [2]), 'D2 end': group_share(b1, f1, [2]),
        'D3-D10 start': group_share(b0, f0, range(3, 11)), 'D3-D10 end': group_share(b1, f1, range(3, 11)),
        'D2-D10 start': group_share(b0, f0, range(2, 11)), 'D2-D10 end': group_share(b1, f1, range(2, 11)),
        'D1 people start': d1_people(b0, f0), 'D1 people end': d1_people(b1, f1),
    })

blog_change = pd.DataFrame(rows).set_index('indicator')
print('World pop 2023 in source units:', global_deciles(2023)['pop'].sum())


def decile_mean(year):
    bins = global_deciles(year)
    return bins.groupby('decile').apply(lambda df: np.average(df['welf'], weights=df['pop']))


income = pd.DataFrame({year: decile_mean(year) for year in (1990, 2000, 2015, 2025)})
income_check = pd.DataFrame({
    '1990-2025 (%)': 100 * (income[2025] / income[1990] - 1),
    '2000-2025 (%)': 100 * (income[2025] / income[2000] - 1),
    '2015-2025 (%)': 100 * (income[2025] / income[2015] - 1),
})

display(blog_change.round(1), income.round(2), income_check.round(1))

World pop 2023 in source units: 8064.057929997226


,start,end,D1 start,D1 end,D2 start,D2 end,D3-D10 start,D3-D10 end,D2-D10 start,D2-D10 end,D1 people start,D1 people end
indicator,,,,,,,,,,,,
Electricity,2000,2023,38.7,30.6,21.7,7.7,10.8,1.2,12.0,1.9,236.8,246.4
Internet,2000,2024,100.0,74.0,100.0,33.4,99.3,6.0,99.3,9.0,584.0,545.7
Sanitation,2000,2024,47.3,66.3,55.7,29.8,37.1,5.3,39.3,8.1,288.8,524.4
Fragile or conflict-affected,2005,2025,26.3,55.8,9.9,28.1,5.0,6.4,5.6,8.8,172.7,457.8


,1990,2000,2015,2025
decile,,,,
1,0.89,1.05,1.88,1.96
2,1.46,1.80,3.14,3.81
3,1.93,2.35,4.08,5.18
4,2.46,2.93,5.16,6.53
5,3.13,3.65,6.62,8.29
6,4.16,4.72,8.73,10.95
7,6.28,6.60,12.08,15.35
8,11.73,10.98,18.23,23.24
9,25.51,25.69,32.90,40.20


,1990-2025 (%),2000-2025 (%),2015-2025 (%)
decile,,,
1,118.9,86.6,4.1
2,161.3,111.2,21.1
3,168.7,120.1,26.9
4,165.2,122.9,26.4
5,164.8,127.1,25.3
6,163.5,132.0,25.5
7,144.3,132.4,27.1
8,98.1,111.7,27.5
9,57.6,56.4,22.2


In [62]:
GROUP_LABELS = {'D1': 'Poorest 10%', 'D2': 'Second-poorest 10%', 'D3-D10': 'Other 80%'}
PANEL_TITLES = {
    'Internet': 'Most people are offline',
    'Electricity': 'Most people lack electricity',
    'Sanitation': 'Most people lack basic sanitation',
    'Fragile or conflict-affected': 'Fragile or conflict-affected state',
}

before_today = pd.DataFrame([
    {
        'panel': f"{PANEL_TITLES[indicator]} ({int(row['start'])} → {int(row['end'])})",
        'group': label,
        'period': period,
        'share': row[f'{group} {column}'],
    }
    for indicator, row in blog_change.iterrows()
    for group, label in GROUP_LABELS.items()
    for period, column in (('Before', 'start'), ('Today', 'end'))
])
# Only the poorest decile's "today" bar is orange, so it stands out from the other groups.
before_today['tone'] = np.select(
    [before_today['period'].eq('Before'), before_today['group'].eq('Poorest 10%')],
    ['Before', 'Today (poorest 10%)'],
    'Today (other groups)',
)
BEFORE_TODAY_PANELS = [
    before_today.loc[before_today['panel'].str.startswith(PANEL_TITLES[name]), 'panel'].iloc[0]
    for name in ('Internet', 'Electricity', 'Sanitation', 'Fragile or conflict-affected')
]
TONES = {'Before': '#D9D9D9', 'Today (poorest 10%)': '#D55E00', 'Today (other groups)': '#5F5F5F'}

before_today_base = alt.Chart().encode(
    y=alt.Y('group:N', sort=list(GROUP_LABELS.values()), title=None, axis=alt.Axis(labelFontSize=12, ticks=False, domain=False)),
    yOffset=alt.YOffset('period:N', sort=['Before', 'Today']),
    x=alt.X('share:Q', scale=alt.Scale(domain=[0, 100]), title='Share of group (%)'),
)
before_today_bars = before_today_base.mark_bar().encode(
    color=alt.Color(
        'tone:N',
        sort=list(TONES),
        scale=alt.Scale(domain=list(TONES), range=list(TONES.values())),
        legend=alt.Legend(title=None, orient='top'),
    ),
    tooltip=[
        alt.Tooltip('panel:N', title='Indicator'),
        alt.Tooltip('group:N', title='Group'),
        alt.Tooltip('period:N', title='Period'),
        alt.Tooltip('share:Q', title='Share (%)', format='.1f'),
    ],
)
before_today_labels = (
    before_today_base.transform_calculate(label="format(datum.share, '.0f') + '%'")
    .mark_text(align='left', dx=3, fontSize=11)
    .encode(
        text='label:N',
        color=alt.Color('tone:N', scale=alt.Scale(domain=list(TONES), range=['#8C8C8C', '#D55E00', '#5F5F5F']), legend=None),
    )
)

before_today_panels = (
    alt.layer(before_today_bars, before_today_labels, data=before_today)
    # A step height gives each Before/Today bar its own band; a fixed height collapses them into lines.
    .properties(width=220, height=alt.Step(14))
    .facet(
        facet=alt.Facet(
            'panel:N', sort=BEFORE_TODAY_PANELS, title=None,
            header=alt.Header(labelFontSize=13, labelFontWeight='bold', labelAnchor='start'),
        ),
        columns=2,
    )
    .resolve_scale(color='independent')
)
before_today_note = alt.Chart(pd.DataFrame({'line': [
    'Share of each group living in a country where fewer than half the population has the service, or in a country on the World Bank list of',
    'fragile and conflict-affected situations. Country-level classification, not individual access. The fragility list methodology changed over the period.',
    'Sources: global income distribution (1,000 bins per country); World Bank WDI (electricity; WHO/UNICEF JMP sanitation); ITU (internet); World Bank FCS lists.',
]}).reset_index()).mark_text(align='left', fontSize=10, color='#7F7F7F').encode(
    y=alt.Y('index:O', axis=None),
    text='line:N',
).properties(width=10, height=45)

before_today_chart = (
    alt.vconcat(before_today_panels, before_today_note, spacing=10)
    .properties(
        title=alt.TitleParams(
            "Progress has largely bypassed the world's poorest tenth",
            subtitle=['Share of each group living in a country where most people lack each service, or in a fragile or conflict-affected state'],
            anchor='start',
            fontSize=16,
        )
    )
    .configure_view(stroke=None)
    .configure_axis(grid=False)
)
before_today_chart

alt.VConcatChart(...)

In [63]:
literacy = fetch_indicator('WB_WDI', 'WB_WDI_SE_ADT_LITR_ZS').drop_duplicates(['code', 'year'])


def literacy_snapshot(target, low, high):
    # Literacy comes from sporadic censuses/surveys: take each country's observation closest to the target year.
    window = literacy[literacy['year'].between(low, high)].copy()
    window['distance'] = (window['year'] - target).abs()
    return window.sort_values(['distance', 'year'], ascending=[True, False]).drop_duplicates('code').set_index('code')


def coverage(bins, flagged, deciles):
    group = bins[bins['decile'].isin(deciles)]
    return 100 * group.loc[group['code'].isin(flagged.index), 'pop'].sum() / group['pop'].sum()


literacy_rows = []
for label, target, low, high, dist_year in (('Around 2000', 2000, 1995, 2005, 2000), ('Latest', 2024, 2014, 2024, 2024)):
    snap = literacy_snapshot(target, low, high)
    flagged = snap['indicator_value'].lt(THRESHOLD).astype(float)
    bins = global_deciles(dist_year)
    literacy_rows.append({
        'period': label, 'window': f'{low}-{high}', 'countries': len(snap),
        'median obs year': snap['year'].median(),
        'D1': group_share(bins, flagged, [1]), 'D2': group_share(bins, flagged, [2]),
        'D2-D10': group_share(bins, flagged, range(2, 11)),
        'D1 people (M)': d1_people(bins, flagged),
        'D1 coverage (%)': coverage(bins, flagged, [1]), 'D2-D10 coverage (%)': coverage(bins, flagged, range(2, 11)),
    })

literacy_summary = pd.DataFrame(literacy_rows).set_index('period')
display(literacy_summary.round(1))

# Same comparison restricted to countries observed in both windows, so the country mix is held fixed.
start_snap, end_snap = literacy_snapshot(2000, 1995, 2005), literacy_snapshot(2024, 2014, 2024)
both = start_snap.index.intersection(end_snap.index)
balanced = pd.DataFrame([
    {
        'period': label,
        'countries': len(both),
        'D1': group_share(global_deciles(year), snap.loc[both, 'indicator_value'].lt(THRESHOLD).astype(float), [1]),
        'D2': group_share(global_deciles(year), snap.loc[both, 'indicator_value'].lt(THRESHOLD).astype(float), [2]),
        'D2-D10': group_share(global_deciles(year), snap.loc[both, 'indicator_value'].lt(THRESHOLD).astype(float), range(2, 11)),
        'D1 coverage (%)': coverage(global_deciles(year), snap.loc[both], [1]),
    }
    for label, year, snap in (('Around 2000', 2000, start_snap), ('Latest', 2024, end_snap))
]).set_index('period')
display(balanced.round(1))

low_literacy_latest = end_snap[end_snap['indicator_value'].lt(THRESHOLD)][['year', 'indicator_value']].sort_values('indicator_value')
display(low_literacy_latest.round(1))

,window,countries,median obs year,D1,D2,D2-D10,D1 people (M),D1 coverage (%),D2-D10 coverage (%)
period,,,,,,,,,
Around 2000,1995-2005,173,2000.0,11.2,12.7,9.4,67.1,97.0,81.9
Latest,2014-2024,163,2022.0,9.6,5.5,1.6,68.3,87.6,82.3


,countries,D1,D2,D2-D10,D1 coverage (%)
period,,,,,
Around 2000,142,11.4,12.9,9.9,95.6
Latest,142,7.2,4.1,1.3,82.6


,year,indicator_value
code,,
TCD,2019,30.6
NER,2022,35.6
MLI,2024,35.6
AFG,2021,37.3
GIN,2018,39.6
BFA,2023,41.6
CAF,2019,42.4
SLE,2019,43.6


In [64]:
CANDIDATES = [
    # (label, dataset, indicator, start target, latest target)
    ('Basic sanitation', 'WB_WDI', 'WB_WDI_SH_STA_BASS_ZS', 2000, 2022),
    ('Clean cooking fuels', 'WB_WDI', 'WB_WDI_EG_CFT_ACCS_ZS', 2000, 2022),
    ('Primary completion', 'WB_WDI', 'WB_WDI_SE_PRM_CMPT_ZS', 2000, 2022),
    ('Account ownership (15+)', 'WB_WDI', 'WB_WDI_FX_OWN_TOTL_ZS', 2011, 2021),
]


def nearest_snapshot(values, target, span=3):
    window = values[values['year'].between(target - span, target + span)].copy()
    window['distance'] = (window['year'] - target).abs()
    return window.sort_values(['distance', 'year'], ascending=[True, False]).drop_duplicates('code').set_index('code')


candidate_rows = []
for label, dataset, indicator_id, start, end in CANDIDATES:
    try:
        values = fetch_indicator(dataset, indicator_id).drop_duplicates(['code', 'year'])
    except Exception as error:
        print(f'{label}: could not fetch ({error})')
        continue
    for period, target in (('start', start), ('latest', end)):
        snap = nearest_snapshot(values, target)
        if snap.empty:
            continue
        flagged = snap['indicator_value'].lt(THRESHOLD).astype(float)
        bins = global_deciles(target)
        candidate_rows.append({
            'indicator': label, 'period': period, 'year': target, 'countries': len(snap),
            'D1': group_share(bins, flagged, [1]), 'D2': group_share(bins, flagged, [2]),
            'D2-D10': group_share(bins, flagged, range(2, 11)),
            'D1 people (M)': d1_people(bins, flagged),
            'D1 coverage (%)': coverage(bins, flagged, [1]),
        })

candidate_summary = pd.DataFrame(candidate_rows).set_index(['indicator', 'period'])
display(candidate_summary.round(1))

year  countries    D1    D2  D2-D10  \
indicator               period                                        
Basic sanitation        start   2000        255  47.1  55.3    37.9   
                        latest  2022        251  62.6  26.3     7.8   
Clean cooking fuels     start   2000        240  89.5  93.1    60.7   
                        latest  2022        240  72.7  36.9    13.8   
Primary completion      start   2000        203  18.4   7.5     3.7   
                        latest  2022        218   6.8   2.6     1.1   
Account ownership (15+) start   2011        176  79.3  82.9    54.4   
                        latest  2021        167  60.5  34.1    16.9   

                                D1 people (M)  D1 coverage (%)  
indicator               period                                  
Basic sanitation        start           288.9             99.7  
                        latest          498.6             99.7  
Clean cooking fuels     start           551.1            100.0  
                        latest          580.7            100.0  
Primary completion      start           103.6             91.5  
                        latest           38.7             71.8  
Account ownership (15+) start           539.6             96.1  
                        latest          437.8             91.4

In [65]:
candidate_summary

year  countries         D1         D2  \
indicator               period                                          
Basic sanitation        start   2000        255  47.067507  55.303481   
                        latest  2022        251  62.574436  26.264793   
Clean cooking fuels     start   2000        240  89.482730  93.146540   
                        latest  2022        240  72.711558  36.879639   
Primary completion      start   2000        203  18.393212   7.491841   
                        latest  2022        218   6.750051   2.581368   
Account ownership (15+) start   2011        176  79.324168  82.937499   
                        latest  2021        167  60.450049  34.147316   

                                   D2-D10  D1 people (M)  D1 coverage (%)  
indicator               period                                             
Basic sanitation        start   37.913397     288.921425        99.666200  
                        latest   7.822664     498.605438        99.745407  
Clean cooking fuels     start   60.709456     551.097611        99.995085  
                        latest  13.762271     580.671364        99.967722  
Primary completion      start    3.720586     103.642263        91.488953  
                        latest   1.057757      38.697272        71.763913  
Account ownership (15+) start   54.381451     539.628604        96.086297  
                        latest  16.858860     437.811569        91.442408